Model 0 setup

In [6]:
# parser 0
import json
import xml.etree.ElementTree as ET
import xml.dom.minidom
import re
import sys
from io import StringIO

def remove_invalid_xml_chars(s):
    """
    Remove characters that are invalid in XML 1.0.
    Allowed characters are:
      - #x9 (tab)
      - #xA (newline)
      - #xD (carriage return)
      - #x20 to #xD7FF
      - #xE000 to #xFFFD
      - #x10000 to #x10FFFF
    This function removes characters in the range:
      #x00-#x08, #x0B-#x0C, and #x0E-#x1F.
    """
    return re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', s)

def parse_recording(line_stream, strip_annotations=False):
    """
    Parse an asciinema recording line-by-line from stdin.

    The stream is expected to have a JSON header on the first non-empty line,
    and then one JSON array per line for each terminal event.

    :param line_stream: An iterable stream of input lines (e.g. sys.stdin).
    :param strip_annotations: If True, do not include annotations.
    :return: XML Element (root) of the generated XML tree.
    """
    root = None
    header_parsed = False

    for line in line_stream:
        line = line.strip()
        if not line:
            continue

        if not header_parsed:
            try:
                header = json.loads(line)
            except Exception as e:
                raise ValueError("Error parsing JSON header: " + str(e))

            root = ET.Element("recording")
            for key in ["version", "width", "height", "timestamp"]:
                if key in header:
                    root.set(key, str(header[key]))

            if not strip_annotations and "librecode_annotations" in header:
                annotations_data = header["librecode_annotations"]
                annotations_elem = ET.SubElement(root, "annotations")
                if "layers" in annotations_data:
                    for layer in annotations_data["layers"]:
                        layer_elem = ET.SubElement(annotations_elem, "layer")
                        if "title" in layer:
                            layer_elem.set("title", layer["title"])
                        if "annotations" in layer:
                            for ann in layer["annotations"]:
                                ann_elem = ET.SubElement(layer_elem, "annotation")
                                if "beginning" in ann:
                                    ann_elem.set("beginning", str(ann["beginning"]))
                                if "end" in ann:
                                    ann_elem.set("end", str(ann["end"]))
                                ann_elem.text = remove_invalid_xml_chars(ann.get("text", ""))
            header_parsed = True
            continue

        try:
            event = json.loads(line)
        except Exception as e:
            print("Skipping line (could not parse JSON):", line, file=sys.stderr)
            continue

        if not isinstance(event, list) or len(event) < 3:
            continue

        timestamp, event_type, content = event[0], event[1], event[2]
        if event_type == "i":
            elem = ET.SubElement(root, "user_input")
        elif event_type == "o":
            elem = ET.SubElement(root, "system_output")
        else:
            continue

        elem.set("timestamp", str(timestamp))
        elem.text = remove_invalid_xml_chars(content)

    return root

def prettify_xml(elem):
    """
    Return a pretty-printed XML string for the Element.
    """
    rough_string = ET.tostring(elem, 'utf-8')
    reparsed = xml.dom.minidom.parseString(rough_string)
    return reparsed.toprettyxml(indent="  ")

def parser_0(filename, content):
  root = parse_recording(StringIO(content))
  xml_text = prettify_xml(root)
  print(xml_text)
  return xml_text

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [3]:
import os
from google.colab import userdata

# 1. SETUP ENVIRONMENT & AUTH
hf_token = userdata.get('HF_TOKEN')
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" # Fast Rust-based downloader
model_path = "./my_local_model"

# 2. INSTALL FAST TRANSFER TOOL
!pip install -q hf_transfer

# 3. DOWNLOAD THE MODEL (The Failsafe Part)
print("Downloading model via CLI (Fast Transfer enabled)...")
!huggingface-cli download Jaiccc/model_0_streaming_timestamp \
    --local-dir {model_path} \
    --token {hf_token}

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
.gitattributes: 1.52kB [00:00, 943kB/s]
Download complete. Moving file to my_local_model/.gitattributes
README.md: 3.25kB [00:00, 11.2MB/s]
Download complete. Moving file to my_local_model/README.md
adapter_config.json: 1.20kB [00:00, 8.03MB/s]
Download complete. Moving file to my_local_model/adapter_config.json
adapter_model.safetensors: 100% 262M/262M [00:02<00:00, 88.3MB/s]
Download complete. Moving file to my_local_model/adapter_model.safetensors
chat_template.jinja: 100% 462/462 [00:00<00:00, 2.87MB/s]
Download complete. Moving file to my_local_model/chat_template.jinja
merges.txt: 917kB [00:00, 13.8MB/s]
Download complete. Moving file to my_local_model/merges.txt
special_tokens_map.json: 100% 570/570 [00:00<00:00, 4.72MB/s]
Download complete. Moving file to my_local_model/special_tokens_map.json
tokenizer.json: 7.15MB [00:00, 108MB/s]
Download complete. Moving file to my_local_model/tokenizer.json
t

In [4]:
# 4. LOAD THE MODEL FROM DISK
from unsloth import FastLanguageModel
import torch

# Ensure ModelScope isn't interfering
if "UNSLOTH_USE_MODELSCOPE" in os.environ:
    del os.environ["UNSLOTH_USE_MODELSCOPE"]

max_seq_length = 4096
load_in_4bit = True

print("\nLoading model from local storage...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path, # Path to the folder we just downloaded
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)
print("Model loaded successfully from local disk!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

Loading model from local storage...
==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/170 [00:00<?, ?B/s]

Unsloth: Will load ./my_local_model as a legacy tokenizer.
Unsloth 2026.3.15 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


Model loaded successfully from local disk!


In [8]:
import re
from google.colab import files
from tqdm.notebook import tqdm

SYSTEM_PROMPT = """Your task is to analyze terminal XML logs and determine whether the timestamp in the TARGET LINE belongs to a "new event" or an "old event".

### DEFINITION OF A NEW EVENT:
1. **Explicit Prompts:** The very first `<user_input>` that immediately follows a shell prompt (e.g., `demo@faiserver:~$`).
2. **Phase Transitions:** In automated logs, moving from one major build stage to another (e.g., from 'fai-mirror finished' to 'Copying the nfsroot').
3. **Internal Logic:** Shifts from downloading to processing.

### WHAT IS *NOT* A NEW EVENT (OLD EVENT):
- **User Input / Keystrokes:** A user typing a command, including pressing the Enter key (a newline `\n`), is just the completion of the input phase.
- **Incomplete Tasks:** Continuous system output without a clear phase shift.

CRITICAL INSTRUCTION: You must classify ONLY the timestamp found in the "### TARGET LINE" section. Do NOT extract timestamps from the "### CONTEXT" section. Output only the timestamp and the classification. Do NOT use brackets, periods, explanations, or markdown formatting."""

CHUNK_RE = re.compile(r"<(user_input|system_output)\b[^>]*?(?:/>|>.*?</\1>)", flags=re.DOTALL)
TIME_RE = re.compile(r'timestamp="([\d\.]+)"')

def get_timestamp(chunk_text: str) -> str:
    match = TIME_RE.search(chunk_text)
    return match.group(1) if match else ""

def truncate_single_chunk(raw_text: str, tag_type: str, max_lines: int = 15) -> str:
    if tag_type != "system_output" or raw_text.endswith("/>"):
        return raw_text
    first_close = raw_text.find(">")
    last_open = raw_text.rfind("</")
    if first_close == -1 or last_open == -1: return raw_text

    opening_tag = raw_text[:first_close+1]
    closing_tag = raw_text[last_open:]
    inner_text = raw_text[first_close+1:last_open]

    lines = inner_text.split('\n')
    if len(lines) > max_lines:
        head = '\n'.join(lines[:5])
        tail = '\n'.join(lines[-5:])
        removed = len(lines) - 10
        return f"{opening_tag}{head}\n\n... [TRUNCATED {removed} LINES] ...\n\n{tail}{closing_tag}"
    return raw_text

def compress_context_window(chunks, max_total_lines=25):
    total_lines = sum(len(c["text"].split('\n')) for c in chunks)
    if total_lines <= max_total_lines:
        return "\n".join([c["text"] for c in chunks])

    result = []
    for idx, c in enumerate(chunks):
        raw_text = c["text"]
        if idx < 5 or idx >= len(chunks) - 5:
            result.append(raw_text)
        else:
            if "<system_output" in raw_text and not raw_text.endswith("/>"):
                first_close = raw_text.find(">")
                last_open = raw_text.rfind("</")
                if first_close != -1 and last_open != -1:
                    opening_tag = raw_text[:first_close+1]
                    closing_tag = raw_text[last_open:]
                    result.append(f"{opening_tag}... [TRUNCATED TO SAVE SPACE] ...{closing_tag}")
                else:
                    result.append(raw_text)
            else:
                result.append(raw_text)
    return "\n".join(result)

def model_0(filename, content):
  # Keep track of all timestamps identified as NEW across all uploaded files
  collected_new_events = []
  xml_content = content.decode('utf-8')

  # Parse and apply Phase 1 truncation
  all_chunks = []
  for match in CHUNK_RE.finditer(xml_content):
      tag_type = match.group(1)
      raw_text = match.group(0)

      # PRE-PHASE TRUNCATION: Protect against massive single-line strings
      if len(raw_text) > 4000:
          raw_text = raw_text[:2000] + "\n...[MASSIVE LINE TRUNCATED]...\n" + raw_text[-2000:]

      processed_text = truncate_single_chunk(raw_text, tag_type, max_lines=15)
      ts_str = get_timestamp(processed_text)
      if ts_str:
          all_chunks.append({"ts": ts_str, "text": processed_text})

  WINDOW_SIZE = 15

  if len(all_chunks) == 0:
      print("No valid timestamps found.")

  print(f"Starting inference on ALL {len(all_chunks)} events...\n")
  print("-" * 50)

  for i in tqdm(range(len(all_chunks)), desc=f"Analyzing"):

      start_idx = max(0, i - WINDOW_SIZE + 1)
      window_chunks = all_chunks[start_idx : i + 1]

      context_chunks = window_chunks[:-1]
      target_chunk = window_chunks[-1]
      target_ts = target_chunk["ts"]

      # Format Context
      if len(context_chunks) > 0:
          context_text = compress_context_window(context_chunks, max_total_lines=25)
      else:
          context_text = "No previous context available."

      target_text = target_chunk["text"]

      # ==========================================
      # PHASE 3 TRUNCATION: ABSOLUTE CHARACTER LIMITS
      # ==========================================
      if len(context_text) > 8000:
          context_text = "... [EARLIER CONTEXT TRUNCATED] ...\n" + context_text[-8000:]

      if len(target_text) > 2000:
          target_text = target_text[:1000] + "\n... [TARGET MIDDLE TRUNCATED] ...\n" + target_text[-1000:]

      input_data = f"### CONTEXT (Previous Events):\n{context_text}\n\n### TARGET LINE (Extract and Classify THIS Timestamp):\n{target_text}"
      combined_user_text = f"{SYSTEM_PROMPT}\n\n{input_data}"

      prompt = f"<|im_start|>user<|im_sep|>{combined_user_text}<|im_end|><|im_start|>assistant<|im_sep|>"

      # Tokenize
      inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

      # Emergency Token Slice
      MAX_TOKENS = 4000
      if inputs.input_ids.shape[1] > MAX_TOKENS:
          inputs.input_ids = inputs.input_ids[:, -MAX_TOKENS:]
          inputs.attention_mask = inputs.attention_mask[:, -MAX_TOKENS:]

      # Generate
      outputs = model.generate(
          input_ids=inputs.input_ids,
          attention_mask=inputs.attention_mask,
          max_new_tokens=32,
          use_cache=True,
          temperature=0.1,
          pad_token_id=tokenizer.eos_token_id
      )

      raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

      # Extract prediction
      m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
      model_result = m.group(1).strip().lower() if m else raw_output.split("assistant")[-1].strip().lower()

      if "new" in model_result:
          icon = "🔥 NEW EVENT"
          collected_new_events.append(target_ts) # Save it to the list
      else:
          icon = "⚪ old event"

      print(f"[{i+1:03d}/{len(all_chunks):03d}] TS: {target_ts:<10} | {icon} | Output: '{model_result}'")

  print("\n" + "="*50)
  print(f"✅ Finished processing! Found {len(collected_new_events)} NEW events.")
  print("COLLECTED NEW EVENT TIMESTAMPS:")
  print(collected_new_events)
  print("="*50)


  def restructure_xml_to_events(input_xml_content, output_xml_path, new_event_timestamps):
      # Parse directly from the in-memory string from your previous block
      tree = ET.parse(io.StringIO(input_xml_content))
      root = tree.getroot()

      # Create the new structured root
      new_root = ET.Element(root.tag, root.attrib)
      trigger_timestamps = set(str(ts).strip() for ts in new_event_timestamps)

      current_event = None

      # Iterate and group into <event> tags
      for child in root:
          ts = child.get("timestamp")

          if current_event is None or ts in trigger_timestamps:
              current_event = ET.SubElement(new_root, "event")

          current_event.append(child)

      # Save the file
      new_tree = ET.ElementTree(new_root)
      if hasattr(ET, "indent"):
          ET.indent(new_tree, space="  ", level=0)

      new_tree.write(output_xml_path, encoding="utf-8", xml_declaration=True)
      print(f"✅ Successfully created structured XML: {output_xml_path}")


  # ==========================================
  # REUSE VARIABLES FROM PREVIOUS BLOCK
  # ==========================================

  # Dynamically generate the output filename using the 'filename' variable already in memory
  if filename.endswith(".xml"):
      output_filename = filename.replace(".xml", "_parsed.xml")
  else:
      output_filename = filename + "_parsed.xml"

  # Run the restructuring function using the 'xml_content' and 'collected_new_events' already in memory
  restructure_xml_to_events(xml_content, output_filename, collected_new_events)


Model 1 setup

In [9]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.environ["HF_HOME"] = "/content/drive/MyDrive/huggingface_cache"

!pip install -q vllm lxml

MessageError: Error: credential propagation was unsuccessful

In [14]:
%%writefile annotator.py
import argparse, json, os, re
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from lxml import etree
from vllm import LLM, SamplingParams

# --- A100 Configuration ---
MODEL_ID = os.getenv("MODEL_ID", "openai/gpt-oss-20b")
GPU_UTIL = float(os.getenv("GPU_UTIL", "0.70"))
MAX_MODEL_LEN = int(os.getenv("MAX_MODEL_LEN", "16384"))
DTYPE = os.getenv("DTYPE", "bfloat16")
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "2000")) # Doubled so it doesn't get cut off while thinking
SUMMARY_WORD_LIMIT = int(os.getenv("SUMMARY_WORD_LIMIT", "50"))
TP_SIZE = int(os.getenv("VLLM_TP", "1"))
K_TARGET = 1
N_NEIGH = 20

_GLOBAL_LLM: Optional[LLM] = None

# --- Prompt Engine & Few Shots ---
FEWSHOTS_BLOCK = """
EXAMPLES (for reference only)

EXAMPLE A — Starting a backup subtask (depth=-1)
neighbor_tail:
  - id=0 depth=0  summary="List project directory contents"
  - id=1 depth=0  summary="Inspect size of source and data folders"
currDepth: 0
input xml:
<event>
  <user_input>t</user_input><system_output>t</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>-</user_input><system_output>-</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <user_input>z</user_input><system_output>z</system_output>
  <user_input>f</user_input><system_output>f</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>b</user_input><system_output>b</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <user_input>k</user_input><system_output>k</system_output>
  <user_input>u</user_input><system_output>u</system_output>
  <user_input>p</user_input><system_output>p</system_output>
  <user_input>.</user_input><system_output>.</system_output>
  <user_input>t</user_input><system_output>t</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>s</user_input><system_output>s</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <system_output>Creating backup.tar...</system_output>
</event>
output:
{"annotation": "Create compressed backup archive of source data", "depth": -1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE B — Continuing backup subtask (depth=0)
neighbor_tail:
  - id=2 depth=-1 summary="Create compressed backup archive of source data"
currDepth: -1
input xml:
<event>
  <user_input>l</user_input><system_output>l</system_output>
  <user_input>s</user_input><system_output>s</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>-</user_input><system_output>-</system_output>
  <user_input>l</user_input><system_output>l</system_output>
  <user_input>h</user_input><system_output>h</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>b</user_input><system_output>b</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <user_input>k</user_input><system_output>k</system_output>
  <user_input>u</user_input><system_output>u</system_output>
  <user_input>p</user_input><system_output>p</system_output>
  <user_input>.</user_input><system_output>.</system_output>
  <user_input>t</user_input><system_output>t</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <system_output>-rw-r--r-- 1 user staff 42M backup.tar</system_output>
</event>
output:
{"annotation": "Verify backup archive and check file size", "depth": 0}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE C — Finishing backup subtask (depth=+1)
neighbor_tail:
  - id=2 depth=-1 summary="Create compressed backup archive of source data"
  - id=3 depth=0  summary="Verify backup archive and check file size"
currDepth: -1
input xml:
<event>
  <user_input>m</user_input><user_input>v</user_input><user_input> </user_input>
  <user_input>b</user_input><user_input>a</user_input><user_input>c</user_input>
  <system_output>Moved to archive/</system_output>
</event>
output:
{"annotation": "Move backup to archive folder and complete backup task", "depth": 1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE D — Starting test/debug subtask (depth=-1)
currDepth: 0
input xml:
<event>
  <user_input>p</user_input><user_input>y</user_input><user_input>t</user_input><user_input>e</user_input><user_input>s</user_input><user_input>t</user_input>
  <system_output>===== test session starts =====</system_output>
</event>
output:
{"annotation": "Start pytest test run for project", "depth": -1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE E — Nested editor within environment setup (depth=-1)
currDepth: -1
input xml:
<event>
  <user_input>v</user_input><user_input>i</user_input><user_input>m</user_input>
  <system_output>Opening vim...</system_output>
</event>
output:
{"annotation": "Open config file in vim during environment setup", "depth": -1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE F — Exit editor, stay in parent task (depth=+1)
currDepth: -2
input xml:
<event>
  <user_input>:</user_input><user_input>w</user_input><user_input>q</user_input>
  <system_output>(venv) $</system_output>
</event>
output:
{"annotation": "Save config changes and exit vim", "depth": 1}
"""

@dataclass
class Event:
    idx: int
    xml: str
    depth_xml: Optional[int] = None
    summary_xml: Optional[str] = None

events: List[Event] = []
pred: Dict[int, Dict[str, object]] = {}

def load_events(xml_path: str) -> List[Event]:
    tree = etree.parse(xml_path)
    root = tree.getroot()
    out = []
    for i, ev_el in enumerate(root.xpath("//event")):
        xml_str = etree.tostring(ev_el, encoding="unicode")
        out.append(Event(idx=i, xml=xml_str))
    return out

def compute_curr_depth_upto(idx: int) -> int:
    curr = 0
    for i in range(idx):
        dep = events[i].depth_xml
        if dep is None: continue
        if dep == -1: curr -= 1
        elif dep > 0: curr += dep
    return curr

def make_flush_package(upto_idx: int, K: int = 1, N: int = 10) -> Dict:
    target_idxs = list(range(max(0, upto_idx - K + 1), upto_idx + 1))
    start_neigh = max(0, target_idxs[0] - N)
    neighbor_idxs = list(range(start_neigh, target_idxs[0]))

    def get_sum(i): return events[i].summary_xml if (0 <= i < len(events) and events[i].summary_xml) else "???"
    def get_dep(i): return events[i].depth_xml if (0 <= i < len(events) and events[i].depth_xml is not None) else 999

    neighbor_info = [f"- id={i} depth={get_dep(i)}  summary={get_sum(i)}" for i in neighbor_idxs]
    target_events = [events[i].xml for i in target_idxs if 0 <= i < len(events)]

    return {
        "target_idxs": target_idxs,
        "neighbor_info": neighbor_info,
        "target_events": target_events,
        "currDepth": compute_curr_depth_upto(target_idxs[0]),
    }

def build_instruction(pkg: dict) -> str:
    # --- NEW: AGGRESSIVE MASSIVE EVENT TRUNCATION ---
    MAX_CHARS = 3000 # Roughly 750 tokens
    truncated_targets = []

    for i, xml in zip(pkg["target_idxs"], pkg["target_events"]):
        if len(xml) > MAX_CHARS:
            # Keep the first 1000 chars (the command) and last 2000 chars (the final result)
            snipped_xml = (
                xml[:1000] +
                f"\n\n... [ ✂️ SYSTEM OUTPUT TRUNCATED - HIDDEN {len(xml) - MAX_CHARS} CHARACTERS ] ...\n\n" +
                xml[-2000:]
            )
            truncated_targets.append(f'<target id="{i}">\n{snipped_xml}\n</target>')
        else:
            truncated_targets.append(f'<target id="{i}">\n{xml}\n</target>')

    targets_xml = "\n".join(truncated_targets)

    return f"""<task>
You are an expert terminal session annotator. Identify goals/subgoals and generate concise action summaries.
</task>

<think_first>
- Keep reasoning CONCISE and FOCUSED
- In <think>...</think>: analyze the command, check depth logic, then conclude
- Aim for 2-3 sentences of reasoning maximum
- Skip obvious observations
- Use neighbors ONLY for continuity; do not invent context.
</think_first>

<rules>
- the user's keystrokes appear separately; combine them to form the full command before interpreting it
- depth is an integer (≥ -1); -1 for subevent (new task started), 0 for same level (still doing the same task), >0 to exit levels (ended one or multiple tasks)
- maintain stack invariant: currDepth ≤ 0; if depth == -1 then currDepth -= 1; if depth > 0 then currDepth += depth
- write action-oriented summaries; avoid "user", "they", "typed", "inputs", "enters a command
- depth is relative to the previous events and nothing else"
</rules>

<output_format>
You MUST wrap your reasoning in <think>...</think> tags.
After the closing </think> tag, you MUST output EXACTLY ONE valid JSON object on a new line. Do not output anything after the JSON.
{{"annotation": "<action summary ≤{SUMMARY_WORD_LIMIT} words>", "depth": <integer ≥ -1>}}
</output_format>

DEPTH SEMANTICS:
- depth = -1: STARTING a new subtask (entering deeper level)
- depth = 0:  CONTINUING at same level (ongoing work)
- depth = +1: FINISHING a subtask (returning to parent level)

<examples>
{FEWSHOTS_BLOCK}
</examples>

<inputs>
  <curr_depth>{pkg.get("currDepth", 0)}</curr_depth>
  <target_events>\n{targets_xml}\n  </target_events>
</inputs>"""

def load_model() -> LLM:
    global _GLOBAL_LLM
    if _GLOBAL_LLM is not None: return _GLOBAL_LLM
    _GLOBAL_LLM = LLM(
        model=MODEL_ID, tensor_parallel_size=TP_SIZE, gpu_memory_utilization=GPU_UTIL,
        max_model_len=MAX_MODEL_LEN, trust_remote_code=True, dtype=DTYPE
    )
    return _GLOBAL_LLM

def generate_with_thinking(llm: LLM, messages: List[Dict[str, str]]):
    tokenizer = llm.get_tokenizer()
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    sampling_params = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS, repetition_penalty=1.2)
    outputs = llm.generate([prompt], sampling_params)
    text = outputs[0].outputs[0].text.strip()
    return text, text, 0, 0

def parse_depth_summary_pairs(text: str) -> List[Tuple[int, str]]:
    dec = json.JSONDecoder()
    out = []

    # NEW PARSING LOGIC: Strip away everything before and inside the <think> tags
    if "</think>" in text:
        text = text.split("</think>")[-1].strip()

    i = 0
    while True:
        start = text.find("{", i)
        if start == -1: break
        try:
            obj, end = dec.raw_decode(text, start)
            if isinstance(obj, dict) and "depth" in obj and "annotation" in obj:
                out.append((int(obj["depth"]), str(obj["annotation"])))
            i = end
        except:
            i = start + 1
    return out

def run_flushes(evs: List[Event]) -> Dict:
    global events, pred
    events = evs
    llm = load_model()

    for upto in range(len(events)):
        success = False
        text = ""
        pkg = {}

        # DYNAMIC CONTEXT SHRINKING: Step down the neighbors if the token limit is exceeded
        for attempt_N in [10, 5, 2, 0]:
            pkg = make_flush_package(upto, N=attempt_N)
            instr = build_instruction(pkg)

            try:
                _, text, _, _ = generate_with_thinking(llm, [{"role": "user", "content": instr}])
                success = True
                break  # Inference succeeded, exit the retry loop
            except Exception as e:
                # Catch the context length error and try again with fewer neighbors
                if "context length" in str(e).lower() or "vllmvalidationerror" in str(type(e)).lower():
                    print(f"\n⚠️ EVENT {upto} EXCEEDED 16K TOKENS (N={attempt_N}). Shrinking history and retrying...")
                    continue
                else:
                    raise e  # If it's a completely different error, crash normally

        if not success:
            print(f"\n❌ EVENT {upto} IS TOO LARGE EVEN WITH 0 NEIGHBORS. Falling back to default.")
            text = '{"annotation": "Event payload too large for context window.", "depth": 0}'

        print(f"\n--- RAW AI OUTPUT FOR EVENT {upto} ---")
        print(text)
        print("--------------------------------------\n")

        pairs = parse_depth_summary_pairs(text)
        if pairs:
            depth, ann = pairs[0]
            pred[pkg["target_idxs"][0]] = {"depth": depth, "summary": ann}
            events[pkg["target_idxs"][0]].depth_xml = depth
            events[pkg["target_idxs"][0]].summary_xml = ann
            print(f"✅ SUCCESSFULLY PARSED -> Depth: {depth} | {ann}")
        else:
            print(f"❌ FAILED TO PARSE JSON FROM EVENT {upto}")

    return pred

def model_1(input_xml):
    evs = load_events(input_xml)
    preds = run_flushes(evs)

    collected_events = []
    for idx, v in preds.items():
        collected_events.append(json.dumps({"idx": idx, **v}) + "\n")
    return "".join(collected_events)


Overwriting annotator.py


Endpoint that collects the recording and returns output of the pipeline

In [13]:
!pip install fastapi
!pip install pyngrok
!pip install uvicorn
!pip install loguru
!pip install pydantic-settings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.2 MB/s eta 0:00:00


In [ ]:
import os
import sys

from flask import Flask, request

def init_webhooks(base_url):
    # ... Implement updates necessary so webhooks use `public_url` from ngrok
    pass

def create_app():
    app = Flask(__name__)

    # Initialize your ngrok settings into Flask
    app.config.from_mapping(
        BASE_URL="http://localhost:5000",
        USE_NGROK=True
    )

    if app.config["USE_NGROK"]:
        # Only import pyngrok and install if we're actually going to use it
        from pyngrok import ngrok
        ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

        # Get the dev server port (defaults to 5000 for Flask, can be overridden with `--port`
        # when starting the server
        port = sys.argv[sys.argv.index("--port") + 1] if "--port" in sys.argv else "5000"

        # Open a ngrok tunnel to the dev server
        public_url = ngrok.connect(port).public_url
        print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{port}\"")

        # Update any base URLs or webhooks to use the public ngrok URL
        app.config["BASE_URL"] = public_url
        init_webhooks(public_url)

    # ... Implement Blueprints and the rest of your app

    return app

app = create_app()
@app.route("/", methods = ['POST'])
def receive_recording_generate_inference():
    request_data = request.get_json(force=True)
    parsed_xml = parser_0(request_data["title"], request_data["content"])
    model_0(request_data["title"], parsed_xml.encode("utf-8"))
    if request_data["title"].endswith(".xml"):
        model_0_output = request_data["title"].replace(".xml", "_parsed.xml")
    else:
        model_0_output = request_data["title"] + "_parsed.xml"
    model_1_output = model_1(model_0_output)
    return {"title": request_data["title"], "content": model_1_output}
    # return {"title": request_data["title"], "content": parsed_xml}

if __name__ == "__main__":
    # use_reloader=False is crucial when running ngrok inside the script
    app.run(port=5000, debug=True, use_reloader=False)

 * ngrok tunnel "https://bustled-aisha-cheerfully.ngrok-free.dev" -> "http://127.0.0.1:5000"
 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Mar/2026 19:16:42] "POST /?title="hi"&content="sdifjoidsjfodsjfosijfodsijfods" HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [25/Mar/2026 19:17:19] "POST / HTTP/1.1" 200 -


{'title': 'titile', 'content': 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'}
